# Episode 9 — Batch Rendering: What Is Hard in Max

⏱ About 25 minutes · Difficulty ★★☆

**Musical element this episode controls: nothing new.** This episode is about **quantity**.

**The Max equivalent:** imagine randomly setting 14 knobs in Max, recording 1 second, saving a file, writing down the knob values, and doing that 10000 times. Possible, but awkward. In Python it is a dozen lines.

**What you will hear:** 20 sounds "dialled in" at random by the computer. Some are nice, some are odd, some are nearly silent — that is normal.

**By the end you can:**
- use a random number generator and make results reproducible
- explain why cutoff and time parameters are randomised "by ratio"
- render in bulk, save wavs, save parameters
- load everything back and verify that "sound ↔ parameters" really are pairs
- arrange a batch of sounds into two tables

**Why it matters:** every sound comes with a record of the parameters that produced it. Such **sound ↔ parameter** pairs are exactly the data needed to train a machine learning model (for example "listen to a sound, guess the synth parameters").

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

import os
import json
import time
from scipy.io import wavfile

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # stack into a 2-D array: F rows × N columns

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # interpolate inside each frame first, then between the two frames
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # times must not be 0, or we would divide by zero below
    # while held: (0,0) → (a,1) → (a+d,s), then stays at s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # how high the envelope is at the moment of release (attack may not be finished)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # fall from that level to 0 over r seconds; np.clip stops it going negative
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # held before release, released after

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff may be one number or an array: make it an array as long as the sound
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- turn cutoff / res into the filter's internal coefficients (one set per sample) ---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping: higher res → smaller k → stronger resonance
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() turns arrays into plain lists: much faster to read one by one in a loop
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # two state variables = two history objects in gen~

    for n in range(n_samples):                 # one sample at a time, in order
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 is the lowpass output
        ic1 = 2 * v1 - ic1                     # update the state for the next sample
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

def phasor(freq, n_samples):
    # freq / sr = how far phase advances during one sample
    # cumsum adds the steps up; % 1 keeps the fractional part
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

def mtof(m):
    return 440 * 2 ** ((m - 69) / 12)          # MIDI 69 = 440 Hz; doubles every 12 semitones

def synth(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr

    # two envelopes
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)

    # oscillator: the wt_pos knob plus what the mod env adds
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"] + p["wt_env"] * mod_env)

    # filter: the cutoff knob, pushed up by cut_env octaves by the mod env
    x = lowpass(x, p["cutoff"] * 2 ** (p["cut_env"] * mod_env), p["res"])

    return x * amp_env                         # VCA

frames = make_frames()

## 1. Random numbers

`rng` is a random number generator. The Max counterparts are `random` and `noise~`.

- `rng.uniform(a, b)`: a random number between a and b, every position equally likely
- `rng.integers(a, b)`: a random whole number from a to b - 1

Every run gives different numbers:

In [ ]:
rng = np.random.default_rng()

print(rng.uniform(0, 1))
print(rng.uniform(0, 1))
print(rng.integers(28, 61))

### The seed: randomness you can repeat

In an experiment "different every time" is a nuisance: you cannot go back and regenerate the same batch of sounds.

Give the generator a **seed** (any whole number) and it produces **the same** random sequence every time. Another seed gives another sequence.

The first two generators below both use seed 0, so their output is identical. The third uses seed 1 and differs.

In [ ]:
first = np.random.default_rng(seed=0)
second = np.random.default_rng(seed=0)
other = np.random.default_rng(seed=1)

print("seed 0:", np.round(first.uniform(0, 1, 4), 3))
print("seed 0:", np.round(second.uniform(0, 1, 4), 3))
print("seed 1:", np.round(other.uniform(0, 1, 4), 3))

## 2. How to randomise a cutoff

The cutoff ranges from 80 to 8000 Hz. Would a plain `rng.uniform(80, 8000)` do?

Episode 7's conclusion: the ear hears frequency by **ratio**. 80 to 8000 is about 6.6 octaves. But drawing evenly in Hz:

- 4000–8000 Hz (the top **one** octave) takes up half the range
- 80–160 Hz (the bottom octave) takes up just 1%

The result: most random sounds come out bright and hardly any dull.

The fix: draw evenly **on a log scale** and convert back. This is log-uniform, the same idea as `geomspace`.

- `np.log(lo)`, `np.log(hi)`: put both ends on a log scale
- draw evenly between the log values
- `np.exp(...)`: convert back to the original unit

Below, 2000 draws of each as a histogram (each bar = how many fell into that interval). The horizontal axis is laid out by octaves.

**You should see:** the left plot (plain uniform) crowded almost entirely to the right; the right plot (log-uniform) roughly equal in every octave.

In [ ]:
rng = np.random.default_rng(seed=0)

def log_uniform(lo, hi):
    # draw evenly between the log values, then convert back with exp
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

plain = []
logged = []
for i in range(2000):
    plain.append(rng.uniform(80, 8000))
    logged.append(log_uniform(80, 8000))

bins = np.geomspace(80, 8000, 30)              # 30 intervals spaced by ratio
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
axes[0].hist(plain, bins=bins)
axes[0].set_title("uniform(80, 8000)")
axes[1].hist(logged, bins=bins)
axes[1].set_title("log_uniform(80, 8000)")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("cutoff (Hz)")
plt.show()

**The rule:**

- frequency and time parameters (`cutoff`, attack, decay, release) → `log_uniform`
- parameters that already run 0 to 1 (`wt_pos`, `res`, sustain, amounts) → plain `uniform`

## 3. Randomising a whole set of parameters

14 parameters, each drawn by the rule above. `float(...)` and `int(...)` turn numpy numbers into plain Python numbers, which is needed for saving later.

The `rng` is created afresh here so that this cell gives the same result on every run.

In [ ]:
rng = np.random.default_rng(seed=0)

def random_params():
    return dict(
        note=int(rng.integers(28, 61)),            # MIDI 28..60
        wt_pos=float(rng.uniform(0, 1)),
        wt_env=float(rng.uniform(0, 1)),
        cutoff=log_uniform(80, 8000),              # a frequency → by ratio
        res=float(rng.uniform(0, 0.9)),
        cut_env=float(rng.uniform(0, 5)),
        amp_a=log_uniform(0.002, 0.5),             # a time → by ratio
        amp_d=log_uniform(0.02, 0.5),
        amp_s=float(rng.uniform(0, 1)),
        amp_r=log_uniform(0.02, 0.4),
        mod_a=log_uniform(0.002, 0.5),
        mod_d=log_uniform(0.02, 0.5),
        mod_s=float(rng.uniform(0, 1)),
        mod_r=log_uniform(0.02, 0.4),
    )

p = random_params()
for name in p:
    print(f"{name:8} = {p[name]:.3f}")

`f"{name:8} = {p[name]:.3f}"` is an **f-string**: put `f` before the quotes and whatever sits in curly braces is replaced by its value. `:8` means "8 characters wide" (for alignment), `:.3f` means "3 decimals".

Listen to this set. One second long, held for 0.6 seconds.

**You should hear:** a random sound — not necessarily a nice one.

In [ ]:
play(synth(p, dur=1.0, gate=0.6))

## 4. Rendering in bulk

Each round:

1. pick a random set of parameters
2. render 1 second with `synth()`
3. save it as a `.wav`
4. append the parameters and the sound to a list each

At the end all parameters go into one `params.json`.

- `os.makedirs("dataset", exist_ok=True)`: create a `dataset` folder next to the notebook (skipped if it exists)
- `f"{i:03d}.wav"`: turn the number `i` into a three-digit file name — `000.wav`, `001.wav` ...
- `os.path.join(a, b)`: join a folder name and a file name into a path
- **JSON** is the standard text format for a `dict` or list; `json.dump` writes, `json.load` reads

In [ ]:
rng = np.random.default_rng(seed=0)            # fixed seed: the same 20 sounds on every run

n_sounds = 20
out_dir = "dataset"
os.makedirs(out_dir, exist_ok=True)

all_params = []
sounds = []
start = time.time()

for i in range(n_sounds):
    p = random_params()                                        # 1. turn the knobs at random
    x = synth(p, dur=1.0, gate=0.6)                            # 2. render
    audio = np.clip(0.3 * x, -1, 1).astype(np.float32)         # lower the level, guard against clipping, convert to the wav format
    wavfile.write(os.path.join(out_dir, f"{i:03d}.wav"), sr, audio)     # 3. save
    all_params.append(p)                                       # 4. record the parameters
    sounds.append(x)

with open(os.path.join(out_dir, "params.json"), "w") as f:
    json.dump(all_params, f, indent=2)

elapsed = time.time() - start
print(f"rendered {n_sounds} sounds in {elapsed:.1f} s")
print(f"at this speed, 10000 sounds would take about {10000 * elapsed / n_sounds / 60:.0f} minutes")

`with open(...) as f:` means "open the file → do the indented things → close it automatically".

Look in the notebook's folder: `dataset/` should contain `000.wav` to `019.wav` and a `params.json`. Open `params.json` in a text editor to see each sound's parameters.

## 5. Listening and looking

### The first 6

**You should hear:** 6 short sounds that differ a lot — in pitch, brightness and length. A few key parameters are printed above each; try matching what you hear to the numbers: higher `note` is higher in pitch, higher `cutoff` is brighter, higher `amp_a` fades in more slowly.

In [ ]:
for i in range(6):
    p = all_params[i]
    print(f"{i:03d}  note={p['note']}  cutoff={p['cutoff']:.0f} Hz  res={p['res']:.2f}  "
          f"cut_env={p['cut_env']:.1f} oct  amp_a={p['amp_a']:.3f} s")
    display(play(sounds[i]))

### All 20 at a glance

Each panel is the spectrogram of one sound (time across, frequency up, brighter = louder).

**You should see:** 20 quite different pictures — some with only a few lines at the bottom (dull), some filled (bright), some with just a patch at the start (short), some with a clear downward-closing shape (filter envelope).

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 7), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    with np.errstate(divide="ignore"):         # fully silent parts trigger a warning when converted to dB; ignore it here
        ax.specgram(sounds[i] + 1e-9, NFFT=1024, Fs=sr, noverlap=768, cmap="magma", vmin=-120)
    ax.set_ylim(0, 6000)
    ax.set_title(f"{i:03d}", fontsize=8)
fig.supxlabel("time (s)")
fig.supylabel("frequency (Hz)")
plt.tight_layout()
plt.show()

## 6. Loading back and verifying the pairs

What does "sound ↔ parameters are pairs" mean? It means: **re-rendering from the saved parameters should give exactly the sound in the saved wav.**

Check number 3:

1. read parameter set 3 from `params.json`
2. run `synth()` on it again
3. read the sound from `003.wav`
4. compare the two

`np.allclose(a, b, atol=1e-6)`: are two arrays "almost equal" (every number within one millionth)?

In [ ]:
with open(os.path.join(out_dir, "params.json")) as f:
    loaded_params = json.load(f)

p3 = loaded_params[3]
rerendered = np.clip(0.3 * synth(p3, dur=1.0, gate=0.6), -1, 1)       # the same processing as when saving

file_sr, from_file = wavfile.read(os.path.join(out_dir, "003.wav"))

print("sample rate in file :", file_sr)
print("same length         :", len(rerendered) == len(from_file))
print("same sound          :", np.allclose(rerendered, from_file, atol=1e-6))

`same sound: True` shows that the parameters describe the sound completely. That is what makes this dataset valuable.

## 7. Quality check

Random sounds vary a lot in loudness and some come out very quiet (a very low cutoff with a low sustain, for instance). When building a dataset you normally want to find those.

**RMS** is one way to compute "average loudness": square each sample → average → square root.

**You should see:** 20 bars of varying height. Anything under the red line counts as "too quiet".

In [ ]:
rms = []
for x in sounds:
    rms.append(np.sqrt(np.mean(x ** 2)))       # square → mean → square root
rms = np.array(rms)

threshold = 0.1

plt.figure(figsize=(9, 3))
plt.bar(np.arange(n_sounds), rms)
plt.axhline(threshold, color="red")
plt.xlabel("sound number")
plt.ylabel("RMS")
plt.xticks(np.arange(n_sounds))
plt.show()

quiet = np.where(rms < threshold)[0]           # the numbers of those whose RMS is below the threshold
print("too quiet:", quiet)

Listen to the first one listed to confirm it really is quieter than the rest, then look at its parameters and work out why.

In [ ]:
if len(quiet) > 0:
    i = int(quiet[0])
    print("sound", i, " note =", all_params[i]["note"], " cutoff =", round(all_params[i]["cutoff"]),
          "Hz  cut_env =", round(all_params[i]["cut_env"], 1), " amp_s =", round(all_params[i]["amp_s"], 2))
    display(play(sounds[i]))
else:
    print("no sound is below the threshold")

## 8. Arranging everything into two tables

A machine learning model does not understand `dict`s and folders; it understands **tables** (2-D arrays) where each row is one example.

- **X**: the sounds. 20 rows × 44100 columns (each row is a 1-second sound)
- **Y**: the parameters. 20 rows × 14 columns (each row holds that sound's 14 parameters)

**Row i of X and row i of Y belong together.** A "listen and guess the parameters" model learns to compute a row of Y from the same row of X.

In [ ]:
names = list(all_params[0].keys())             # the 14 parameter names, in a fixed order

rows = []
for p in all_params:
    row = []
    for name in names:
        row.append(p[name])                    # take the 14 values in the same order
    rows.append(row)

X = np.array(sounds)
Y = np.array(rows)

print("names:", names)
print("X shape:", X.shape, "  (sounds x samples)")
print("Y shape:", Y.shape, "     (sounds x parameters)")
print()
print("row 3 of Y:", np.round(Y[3], 3))

At this point a (very small) dataset is complete. Change `n_sounds` to 10000 and the same code produces a dataset you could really train on.

## Common mistakes

| Symptom | Cause |
|---|---|
| Different sounds on every run | `rng` was created without a seed, or the line creating it was not re-run |
| `Object of type float32 is not JSON serializable` | A numpy number slipped into the parameters. Wrap it in `float(...)` / `int(...)` |
| `same sound: False` after loading | `dur`, `gate` or the level processing differ between re-rendering and saving |
| The wav files distort in other software | No `np.clip` or level reduction before saving |

## Exercises

**1.** Change the seed to 1 and re-run sections 4 and 5. **You should hear:** 20 completely different sounds. Change it back to 0 and the original batch returns.

**2.** Build a "bass only" dataset: restrict `note` to 28–40 and `cutoff` to 80–1000. Hint: copy `random_params` and change two lines.

<details><summary>Answer</summary>

```python
def random_bass():
    p = random_params()
    p["note"] = int(rng.integers(28, 41))
    p["cutoff"] = log_uniform(80, 1000)
    return p

rng = np.random.default_rng(seed=5)
for i in range(4):
    display(play(synth(random_bass(), dur=1.0, gate=0.6)))
```
</details>

**3.** Find the sound with the highest cutoff in `Y` and play it. Hint: `names.index("cutoff")` gives the column of cutoff; `np.argmax(column)` gives the row of the largest value.

<details><summary>Answer</summary>

```python
col = names.index("cutoff")
i = int(np.argmax(Y[:, col]))          # Y[:, col] = column col of every row
print("sound", i, "cutoff", round(Y[i, col]))
play(sounds[i])
```
</details>

## Recap

| | Max / RNBO | Python |
|---|---|---|
| playing live, turning knobs | excellent | poor |
| auto-generating ten thousand sounds with their parameters | awkward | one loop |

| Max | Python |
|---|---|
| `random` | `rng.uniform(a, b)`, `rng.integers(a, b)` |
| the `seed` message | `np.random.default_rng(seed=0)` |
| `sfrecord~` | `wavfile.write(...)` |
| `write` on `dict` | `json.dump(...)` |

The two do not replace each other; they split the work:

- **Python**: offline, in bulk, reproducible — building data, analysis, experiments
- **Max / RNBO**: real time, interactive — building instruments and things people use

**Next episode (the last):** looking at sound in Python, then sending the finished wavetable and presets back to Max.